# Telco Genie Learning Day — Synthetic Dataset Generator
**Generates 5 tables** into Unity Catalog for the Genie demo & workshop.

| Table | Description |
|-------|-------------|
| `plans` | 6 mobile plan tiers (Essential → Premium) |
| `customers` | ~10K customers with churn signals, metro/regional split |
| `usage` | Monthly data/calls/SMS usage over 3 months (Jan–Mar 2026) |
| `support_tickets` | CX-focused: complaints, NPS, resolution times |
| `network_events` | Outages & maintenance, skewed to regional areas |

**Key data stories baked in:**
- Regional vs metro gap (churn, resolution time, NPS, network events)
- Clear churn signals (tenure, complaint frequency, low NPS, prepaid)
- Australian-flavoured: Australian cities, AUD, plan names echoing real telco tiers

## Configuration
Set your target catalog and schema below.

In [0]:
CATALOG = "workspace"
SCHEMA = "telco"

# spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"USE {CATALOG}.{SCHEMA}")

print(f"Target: {CATALOG}.{SCHEMA}")

## Reference Data

In [0]:
import pyspark.sql.functions as F
import pyspark.sql.types as T
from pyspark.sql import Row
from datetime import date, timedelta
import random
import numpy as np

random.seed(42)
np.random.seed(42)

# Australian states with metro/regional cities
STATES = {
    "NSW": {"metro": ["Sydney"], "regional": ["Newcastle", "Wollongong", "Coffs Harbour", "Tamworth"]},
    "VIC": {"metro": ["Melbourne"], "regional": ["Geelong", "Ballarat", "Bendigo", "Shepparton"]},
    "QLD": {"metro": ["Brisbane"], "regional": ["Gold Coast", "Townsville", "Cairns", "Rockhampton"]},
    "WA":  {"metro": ["Perth"], "regional": ["Bunbury", "Geraldton", "Kalgoorlie"]},
    "SA":  {"metro": ["Adelaide"], "regional": ["Mount Gambier", "Whyalla", "Port Augusta"]},
    "TAS": {"metro": ["Hobart"], "regional": ["Launceston", "Devonport"]},
    "NT":  {"metro": ["Darwin"], "regional": ["Alice Springs", "Katherine"]},
    "ACT": {"metro": ["Canberra"], "regional": []},
}

STATE_WEIGHTS = {"NSW": 0.32, "VIC": 0.26, "QLD": 0.20, "WA": 0.10, "SA": 0.07, "TAS": 0.02, "NT": 0.01, "ACT": 0.02}

PLAN_IDS =     ["BAS-30",  "BAS-60",  "PLU-80",  "PLU-120", "PRM-200", "PRM-UNL"]
PLAN_WEIGHTS = [0.20,      0.25,      0.22,      0.15,      0.12,      0.06]

PLAN_COST = {"BAS-30": 35, "BAS-60": 50, "PLU-80": 65, "PLU-120": 85, "PRM-200": 115, "PRM-UNL": 150}
PLAN_DATA = {"BAS-30": 30, "BAS-60": 60, "PLU-80": 80, "PLU-120": 120, "PRM-200": 200, "PRM-UNL": 9999}

TICKET_CATEGORIES = {
    "Billing":         ["Incorrect charge", "Payment failed", "Plan change issue", "Overage dispute"],
    "Network":         ["No coverage", "Slow data speed", "Dropped call", "Outage report"],
    "Account":         ["Login issue", "Update details", "Cancel service", "Transfer number"],
    "Device":          ["Setup help", "Handset fault", "SIM issue", "5G compatibility"],
    "General Enquiry": ["Plan comparison", "Coverage check", "International roaming", "New connection"],
}

CHANNELS = ["App", "Phone", "Web Chat", "In-Store", "Social Media"]
CHANNEL_WEIGHTS = [0.30, 0.25, 0.25, 0.12, 0.08]

SEVERITY_LEVELS = ["Low", "Medium", "High", "Critical"]

EVENT_TYPES = ["Planned Maintenance", "Unplanned Outage", "Degraded Performance", "Capacity Upgrade"]

DATE_START = date(2026, 1, 1)
DATE_END = date(2026, 3, 31)
MONTHS = ["2026-01", "2026-02", "2026-03"]

NUM_CUSTOMERS = 10_000

## Helper Functions

In [0]:
def pick_location():
    """Return (state, city, region_type) with ~65% metro, ~35% regional."""
    state = random.choices(list(STATE_WEIGHTS.keys()), weights=list(STATE_WEIGHTS.values()))[0]
    info = STATES[state]
    if info["regional"] and random.random() < 0.35:
        return state, random.choice(info["regional"]), "Regional"
    else:
        return state, random.choice(info["metro"]), "Metro"

def random_date(start, end):
    return start + timedelta(days=random.randint(0, (end - start).days))

## 1. Plans

In [0]:
plans_data = [
    ("BAS-30",  "Essential 30GB",    "Prepaid",  35,  30,   999),
    ("BAS-60",  "Essential 60GB",    "Prepaid",  50,  60,   999),
    ("PLU-80",  "Plus 80GB",         "Postpaid", 65,  80,   999),
    ("PLU-120", "Plus 120GB",        "Postpaid", 85,  120,  999),
    ("PRM-200", "Premium 200GB",     "Postpaid", 115, 200,  999),
    ("PRM-UNL", "Premium Unlimited", "Postpaid", 150, 9999, 999),
]

df_plans = spark.createDataFrame(plans_data, ["plan_id", "plan_name", "plan_category", "monthly_cost_aud", "data_allowance_gb", "included_calls_min"])
df_plans.write.mode("overwrite").saveAsTable("plans")
display(spark.table("plans"))

## 2. Customers
~10K customers with realistic churn signals:
- **Regional churn ~4%** vs **metro ~1.8%**
- Prepaid, short tenure, and 4G devices increase churn probability

In [0]:
customers = []
for i in range(1, NUM_CUSTOMERS + 1):
    state, city, region_type = pick_location()
    plan_id = random.choices(PLAN_IDS, weights=PLAN_WEIGHTS)[0]
    signup_date = random_date(date(2021, 1, 1), date(2025, 12, 31))
    device_type = random.choices(["4G", "5G"], weights=[0.45, 0.55])[0]

    # Churn: regional ~4%, metro ~1.8%, boosted by tenure/plan/device
    base_churn = 0.04 if region_type == "Regional" else 0.018
    tenure_days = (DATE_END - signup_date).days
    if tenure_days < 365:
        base_churn *= 1.5
    if plan_id.startswith("BAS"):
        base_churn *= 1.3
    if device_type == "4G":
        base_churn *= 1.1

    is_churned = random.random() < min(base_churn, 0.15)
    churn_date = random_date(DATE_START, DATE_END) if is_churned else None
    status = "Churned" if is_churned else "Active"

    customers.append(Row(
        customer_id=f"CUST-{i:06d}",
        state=state,
        city=city,
        region_type=region_type,
        signup_date=signup_date,
        plan_id=plan_id,
        device_type=device_type,
        status=status,
        churn_date=churn_date,
    ))

df_customers = spark.createDataFrame(customers)
df_customers.write.mode("overwrite").saveAsTable("customers")

# Quick stats
total = df_customers.count()
churned = df_customers.filter(F.col("status") == "Churned").count()
print(f"Total: {total:,} | Churned: {churned:,} ({churned/total*100:.1f}%)")
display(spark.sql("SELECT region_type, status, COUNT(*) AS count FROM customers GROUP BY region_type, status ORDER BY region_type, status"))

## 3. Monthly Usage
3 months (Jan–Mar 2026) per active customer. Regional users tend to use slightly less data.

In [0]:
usage_rows = []
for c in customers:
    if c.status == "Churned" and c.churn_date:
        active_months = [m for m in MONTHS if m <= c.churn_date.strftime("%Y-%m")]
    else:
        active_months = MONTHS

    allowance = PLAN_DATA[c.plan_id]

    for month in active_months:
        usage_factor = 0.85 if c.region_type == "Regional" else 1.0
        if allowance >= 9999:
            data_used = max(0.0, float(np.random.normal(80, 30) * usage_factor))
        else:
            data_used = max(0.0, float(np.random.normal(allowance * 0.7, allowance * 0.2) * usage_factor))

        calls_min = max(0, int(np.random.normal(120, 60)))
        sms_count = max(0, int(np.random.normal(30, 20)))

        overage_gb = max(0.0, data_used - allowance) if allowance < 9999 else 0.0
        overage_charges = round(overage_gb * 10, 2)

        usage_rows.append(Row(
            customer_id=c.customer_id,
            month=month,
            data_used_gb=round(data_used, 2),
            calls_min=calls_min,
            sms_count=sms_count,
            overage_charges_aud=overage_charges,
        ))

df_usage = spark.createDataFrame(usage_rows)
df_usage.write.mode("overwrite").saveAsTable("usage")
print(f"Usage rows: {df_usage.count():,}")
display(spark.table("usage").limit(10))

## 4. Support Tickets
The star table for CX analysis. Churned customers have more tickets, regional customers get more network complaints and slower resolution times.

In [0]:
ticket_id = 0
ticket_rows = []

for c in customers:
    if c.status == "Churned":
        num_tickets = np.random.poisson(4)
    elif c.region_type == "Regional":
        num_tickets = np.random.poisson(1.8)
    else:
        num_tickets = np.random.poisson(1.0)

    for _ in range(int(num_tickets)):
        ticket_id += 1

        # Regional skews toward Network tickets
        if c.region_type == "Regional":
            cat_weights = [0.20, 0.35, 0.15, 0.15, 0.15]
        else:
            cat_weights = [0.25, 0.15, 0.20, 0.20, 0.20]

        category = random.choices(list(TICKET_CATEGORIES.keys()), weights=cat_weights)[0]
        subcategory = random.choice(TICKET_CATEGORIES[category])
        channel = random.choices(CHANNELS, weights=CHANNEL_WEIGHTS)[0]

        if category == "Network":
            severity = random.choices(SEVERITY_LEVELS, weights=[0.15, 0.30, 0.35, 0.20])[0]
        else:
            severity = random.choices(SEVERITY_LEVELS, weights=[0.35, 0.35, 0.20, 0.10])[0]

        created_date = random_date(DATE_START, DATE_END)

        base_hours = {"Low": 4, "Medium": 12, "High": 24, "Critical": 48}[severity]
        region_factor = 1.4 if c.region_type == "Regional" else 1.0
        resolution_hours = round(max(0.5, float(np.random.exponential(base_hours * region_factor))), 1)

        if c.status == "Churned":
            nps = max(0, min(10, int(np.random.normal(3, 2))))
        elif c.region_type == "Regional":
            nps = max(0, min(10, int(np.random.normal(6, 2))))
        else:
            nps = max(0, min(10, int(np.random.normal(7.5, 1.5))))

        ticket_rows.append(Row(
            ticket_id=f"TKT-{ticket_id:06d}",
            customer_id=c.customer_id,
            created_date=created_date,
            category=category,
            subcategory=subcategory,
            severity=severity,
            channel=channel,
            resolution_time_hours=resolution_hours,
            nps_score=nps,
        ))

df_tickets = spark.createDataFrame(ticket_rows)
df_tickets.write.mode("overwrite").saveAsTable("support_tickets")
print(f"Support tickets: {df_tickets.count():,}")
display(spark.sql("""
    SELECT category, COUNT(*) AS ticket_count,
           ROUND(AVG(nps_score), 1) AS avg_nps,
           ROUND(AVG(resolution_time_hours), 1) AS avg_resolution_hrs
    FROM support_tickets GROUP BY category
"""))

## 5. Network Events
~200 events over 3 months. 60% occur in regional areas with longer durations.

In [0]:
event_rows = []
for i in range(1, 201):
    state = random.choices(list(STATE_WEIGHTS.keys()), weights=list(STATE_WEIGHTS.values()))[0]
    info = STATES[state]

    if info["regional"] and random.random() < 0.60:
        city = random.choice(info["regional"])
        region_type = "Regional"
    else:
        city = random.choice(info["metro"])
        region_type = "Metro"

    event_type = random.choices(EVENT_TYPES, weights=[0.30, 0.25, 0.30, 0.15])[0]
    event_date = random_date(DATE_START, DATE_END)

    if event_type == "Unplanned Outage":
        base_duration = 180 if region_type == "Regional" else 90
    elif event_type == "Degraded Performance":
        base_duration = 120 if region_type == "Regional" else 60
    else:
        base_duration = 60

    duration_min = max(10, int(np.random.exponential(base_duration)))
    affected_customers = max(50, int(np.random.exponential(500 if region_type == "Metro" else 150)))

    event_rows.append(Row(
        event_id=f"EVT-{i:04d}",
        event_date=event_date,
        state=state,
        city=city,
        region_type=region_type,
        event_type=event_type,
        duration_min=duration_min,
        affected_customers=affected_customers,
    ))

df_events = spark.createDataFrame(event_rows)
df_events.write.mode("overwrite").saveAsTable("network_events")
print(f"Network events: {df_events.count():,}")
display(spark.sql("SELECT region_type, event_type, COUNT(*) AS count FROM network_events GROUP BY region_type, event_type ORDER BY region_type, event_type"))

## Validation Queries
Quick checks to confirm the data tells the right stories.

In [0]:
%sql
-- Churn rate by region type
SELECT
  region_type,
  COUNT(*) AS total_customers,
  SUM(CASE WHEN status = 'Churned' THEN 1 ELSE 0 END) AS churned,
  ROUND(SUM(CASE WHEN status = 'Churned' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) AS churn_rate_pct
FROM customers
GROUP BY region_type
ORDER BY region_type

In [0]:
%sql
-- ARPU by plan (Average Revenue Per User)
SELECT
  p.plan_name,
  p.monthly_cost_aud,
  COUNT(DISTINCT c.customer_id) AS subscribers,
  ROUND(AVG(p.monthly_cost_aud + COALESCE(u.overage_charges_aud, 0)), 2) AS arpu
FROM customers c
JOIN plans p ON c.plan_id = p.plan_id
LEFT JOIN usage u ON c.customer_id = u.customer_id
GROUP BY p.plan_name, p.monthly_cost_aud
ORDER BY p.monthly_cost_aud

In [0]:
%sql
-- NPS by region — the gap should be visible
SELECT
  c.region_type,
  ROUND(AVG(t.nps_score), 1) AS avg_nps,
  COUNT(*) AS ticket_count,
  ROUND(AVG(t.resolution_time_hours), 1) AS avg_resolution_hrs
FROM support_tickets t
JOIN customers c ON t.customer_id = c.customer_id
GROUP BY c.region_type

In [0]:
%sql
-- Top complaint subcategories
SELECT
  category,
  subcategory,
  COUNT(*) AS ticket_count,
  ROUND(AVG(nps_score), 1) AS avg_nps
FROM support_tickets
GROUP BY category, subcategory
ORDER BY ticket_count DESC
LIMIT 10

In [0]:
%sql
-- Network events: regional vs metro
SELECT
  region_type,
  event_type,
  COUNT(*) AS event_count,
  ROUND(AVG(duration_min), 0) AS avg_duration_min,
  ROUND(AVG(affected_customers), 0) AS avg_affected
FROM network_events
GROUP BY region_type, event_type
ORDER BY region_type, event_count DESC

## Done!

Tables created in `workspace.telco`:
- `plans` — 6 rows
- `customers` — ~10,000 rows
- `usage` — ~29,000 rows
- `support_tickets` — ~15,000 rows
- `network_events` — 200 rows

**Next step:** Run `02_uc_documentation_and_constraints.py` (Unity Catalog comments + PK/FK), then build the Genie space.